# vLLM vs Transformers Benchmark (Electronics Few-Shot)

Experimental benchmark notebook to compare throughput on a deterministic Electronics subset.

- Production workflow stays unchanged.
- Uses the same prompt path from `src/carbon/retrieval.py`.
- Saves benchmark artifacts under Drive so this can be removed safely if not useful.



## 1) Mount Drive


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2) Setup Repo and Dependencies

Installs project requirements and attempts to install `vllm` (non-fatal if unavailable).


In [ ]:
from pathlib import Path
import subprocess
import sys
import shutil

DRIVE_BASE = Path('/content/drive/MyDrive')
REPO_ROOT = Path('/content/carbon-aware-recsys')
REPO_URL = 'https://github.com/andersvestrum/carbon-aware-recsys.git'
REPO_BRANCH = 'last-run'

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)],
    check=True,
)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'accelerate'], check=True)

vllm_available = True
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)
except Exception as exc:
    vllm_available = False
    print(f'vLLM install failed: {exc}')

print({'repo_root': str(REPO_ROOT), 'repo_branch': REPO_BRANCH, 'vllm_available': vllm_available})


## 3) Benchmark Configuration


In [ ]:
import os
import sys
import time
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(REPO_ROOT))

from src.data.amazon_loader import load_meta
from src.carbon.retrieval import (
    OpenAILLMClient,
    TransformersLLMClient,
    PCFRetrievalEstimator,
    RetrievalConfig,
    set_global_determinism,
    parse_numeric_response,
)

LLM_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
CATEGORY = 'electronics'
BENCHMARK_SIZES = [200, 500]
SEED = 42
TOP_K = 5
NUM_THREADS = 4
LLM_REASONING_STYLE = 'terse'
MAX_NEW_TOKENS = 128

# Keep same worker count for both backends in this benchmark for a fair baseline.
LLM_WORKERS = 1
RUN_LABEL = 'vllm_ab'

DRIVE_BENCH_BASE = DRIVE_BASE / 'carbon-aware-recsys-colab' / 'pcf' / 'benchmarks' / 'vllm_vs_transformers'
RUN_TAG = f"electronics_seed{SEED}_sizes{'-'.join(str(x) for x in BENCHMARK_SIZES)}_{RUN_LABEL}"
RUN_DIR = DRIVE_BENCH_BASE / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)

RUN_LOG = RUN_DIR / 'benchmark.log'
logger = logging.getLogger('vllm_benchmark')
logger.setLevel(logging.INFO)
logger.handlers.clear()
fmt = logging.Formatter('%(asctime)s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')
sh = logging.StreamHandler(sys.stdout)
sh.setFormatter(fmt)
fh = logging.FileHandler(RUN_LOG, mode='a', encoding='utf-8')
fh.setFormatter(fmt)
logger.addHandler(sh)
logger.addHandler(fh)

logger.info('Benchmark run dir: %s', RUN_DIR)
print({'run_tag': RUN_TAG, 'run_dir': str(RUN_DIR), 'run_log': str(RUN_LOG)})


## 4) Load Metadata, Build Deterministic Samples


In [ ]:
logger.info('Loading full electronics metadata...')
meta = load_meta(CATEGORY)
max_n = max(BENCHMARK_SIZES)
sample_n = min(max_n, len(meta))

sampled = (
    meta.sample(n=sample_n, random_state=SEED, replace=False)
    .sort_values('parent_asin')
    .reset_index(drop=True)
)

samples = {n: sampled.head(n).copy() for n in BENCHMARK_SIZES}
logger.info('Full rows=%d, sampled rows=%d', len(meta), len(sampled))
for n, df in samples.items():
    logger.info('Prepared benchmark sample n=%d rows=%d', n, len(df))


## 5) Build Retriever Once (Shared)

Retrieval index is built once and reused for both backends to avoid setup skew.


In [ ]:
set_global_determinism(SEED, deterministic=True, num_threads=NUM_THREADS)

config = RetrievalConfig(
    top_k=TOP_K,
    random_seed=SEED,
    deterministic=True,
    num_threads=NUM_THREADS,
    llm_workers=LLM_WORKERS,
    device='cpu',
)

estimator = PCFRetrievalEstimator(config)
logger.info('Fitting Carbon Catalogue retrieval index...')
estimator.fit_carbon_catalogue()
logger.info('Retrieval index ready.')


## 6) vLLM Adapter (Experimental)

Implements the same `predict_numeric(prompt)` interface used by the pipeline.


In [ ]:
class VLLMNumericClient:
    system_prompt = OpenAILLMClient.system_prompt
    terse_system_prompt = OpenAILLMClient.terse_system_prompt

    def __init__(
        self,
        model: str,
        *,
        max_new_tokens: int = 128,
        reasoning_style: str = 'terse',
        tensor_parallel_size: int = 1,
        gpu_memory_utilization: float = 0.9,
    ):
        self.model = model
        self.max_new_tokens = max_new_tokens
        self.reasoning_style = reasoning_style
        self.system_prompt = (
            self.terse_system_prompt if reasoning_style == 'terse' else self.system_prompt
        )
        self.tensor_parallel_size = tensor_parallel_size
        self.gpu_memory_utilization = gpu_memory_utilization
        self._llm = None
        self._tokenizer = None

    @property
    def is_available(self):
        return True

    def _lazy_init(self):
        if self._llm is not None:
            return
        from vllm import LLM
        from transformers import AutoTokenizer

        self._llm = LLM(
            model=self.model,
            tensor_parallel_size=self.tensor_parallel_size,
            gpu_memory_utilization=self.gpu_memory_utilization,
            dtype='float16',
        )
        self._tokenizer = AutoTokenizer.from_pretrained(self.model)

    def predict_numeric(self, prompt: str) -> float:
        self._lazy_init()
        from vllm import SamplingParams

        messages = [
            {'role': 'system', 'content': self.system_prompt},
            {'role': 'user', 'content': prompt},
        ]
        prompt_text = self._tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        params = SamplingParams(
            temperature=0.0,
            max_tokens=self.max_new_tokens,
        )
        out = self._llm.generate([prompt_text], params)
        text = out[0].outputs[0].text
        return parse_numeric_response(text)


## 7) Benchmark Runner


In [ ]:
def run_backend_once(backend_name: str, llm_client, sample_df: pd.DataFrame, n: int) -> dict:
    cache_path = RUN_DIR / f'llm_cache_{backend_name}_n{n}.jsonl'
    pred_path = RUN_DIR / f'predictions_{backend_name}_n{n}.csv'

    t0 = time.time()
    pred_df = estimator.predict_amazon_products(
        sample_df,
        llm_client=llm_client,
        llm_model_name=LLM_MODEL,
        llm_cache_path=cache_path,
        llm_limit=None,
        llm_cache_only=False,
        llm_methods=('few_shot_llm',),
        llm_reasoning_style=LLM_REASONING_STYLE,
    )
    elapsed = time.time() - t0

    pred_df.to_csv(pred_path, index=False)
    parsed = int(pred_df['few_shot_llm_pcf'].notna().sum()) if 'few_shot_llm_pcf' in pred_df.columns else 0
    total = int(len(pred_df))

    result = {
        'backend': backend_name,
        'n': int(n),
        'elapsed_seconds': float(elapsed),
        'elapsed_minutes': float(elapsed / 60.0),
        'items_per_second': float(total / elapsed) if elapsed > 0 else 0.0,
        'seconds_per_item': float(elapsed / total) if total > 0 else float('nan'),
        'parse_success_count': parsed,
        'parse_success_rate': float(parsed / total) if total > 0 else 0.0,
        'predictions_path': str(pred_path),
        'cache_path': str(cache_path),
    }
    logger.info('[%s n=%d] %.2f min | %.3f items/s | parse %.1f%%',
                backend_name, n, result['elapsed_minutes'], result['items_per_second'], 100*result['parse_success_rate'])
    return result


## 8) Execute A/B Benchmark

Runs Transformers baseline first, then vLLM (if available) on the same deterministic sample rows.


In [ ]:
results = []

for n in BENCHMARK_SIZES:
    df_n = samples[n]
    logger.info('--- Benchmark n=%d start ---', n)

    # Baseline transformers client
    transformers_client = TransformersLLMClient(
        model=LLM_MODEL,
        torch_dtype='float16',
        device_map='auto',
        max_new_tokens=MAX_NEW_TOKENS,
        reasoning_style=LLM_REASONING_STYLE,
    )
    results.append(run_backend_once('transformers', transformers_client, df_n, n))

    # Experimental vLLM backend
    if vllm_available:
        try:
            vllm_client = VLLMNumericClient(
                model=LLM_MODEL,
                max_new_tokens=MAX_NEW_TOKENS,
                reasoning_style=LLM_REASONING_STYLE,
            )
            results.append(run_backend_once('vllm', vllm_client, df_n, n))
        except Exception as exc:
            logger.exception('vLLM benchmark failed for n=%d: %s', n, exc)
            results.append({
                'backend': 'vllm',
                'n': int(n),
                'error': str(exc),
            })
    else:
        results.append({
            'backend': 'vllm',
            'n': int(n),
            'error': 'vllm_not_available',
        })

summary_df = pd.DataFrame(results)
summary_path = RUN_DIR / 'benchmark_summary.csv'
summary_df.to_csv(summary_path, index=False)

meta = {
    'run_tag': RUN_TAG,
    'llm_model': LLM_MODEL,
    'benchmark_sizes': BENCHMARK_SIZES,
    'seed': SEED,
    'llm_reasoning_style': LLM_REASONING_STYLE,
    'max_new_tokens': MAX_NEW_TOKENS,
    'llm_workers': LLM_WORKERS,
    'summary_path': str(summary_path),
    'run_log': str(RUN_LOG),
}
with (RUN_DIR / 'benchmark_metadata.json').open('w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

print('Saved benchmark outputs:')
print(' -', summary_path)
print(' -', RUN_DIR / 'benchmark_metadata.json')
display(summary_df)


## 9) Optional Quick Decision Rule

Use this to quickly judge whether vLLM is worth integrating into broader runs.


In [ ]:
if 'summary_df' in globals() and not summary_df.empty:
    view = summary_df[['backend', 'n', 'items_per_second', 'seconds_per_item', 'parse_success_rate']].copy()
    print(view)

    for n in BENCHMARK_SIZES:
        sub = summary_df[(summary_df['n'] == n) & (summary_df['backend'].isin(['transformers', 'vllm']))]
        if len(sub) == 2 and sub['items_per_second'].notna().all():
            t_rate = float(sub[sub['backend'] == 'transformers']['items_per_second'].iloc[0])
            v_rate = float(sub[sub['backend'] == 'vllm']['items_per_second'].iloc[0])
            if t_rate > 0:
                print(f'n={n}: vLLM speedup = {v_rate / t_rate:.2f}x')
else:
    print('Run benchmark cell first.')
